In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

from combra import data
from combra.metrics import (
    metrics_vs_n, find_ethalon,
    convergence_stats, print_convergence_report,
    summarize_metric_distribution,
    plot_wdist_convergence_grid, plot_metric_distribution,
)

# Angle binning step (degrees). Used by generation AND the W-dist computation.
# generate_angles overwrites parquets, so changing STEP and re-running generation
# drops any previously-stored step. Files already containing STEP are skipped.
STEP = 2.0
ANGLES_ROOT = Path('./data/angles')

def sweep_dir(h5_path):
    return ANGLES_ROOT / (Path(h5_path).stem + '_msl5')


In [ ]:
# Per-resolution sources for the 3x3 convergence grid. Each row = one resolution;
# the 'real' H5 doubles as ethalon (largest-N parquet) and as source of the
# original self-convergence curve. Diffit h5s match 4_grid_plot.ipynb's SOURCES
# so wdist values agree with that notebook's table.
SOURCES = {
    256: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
        'san':    './data/h5/gen_san_256x256_N100_000.h5',
        'diffit': './data/h5/00017-diffit-256-gpus2-batch192_N10000.h5',
    },
    512: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
        'san':    './data/h5/gen_san_512x512_N100_000.h5',
        'diffit': './data/h5/00018-diffit-512-gpus4-batch256_N10000.h5',
    },
    1024: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360.h5',
    },
}

# N sweep per (kind, resolution). Bounded by per-class dataset size of each H5 —
# diffit at 512 only has 1k/class so its sweep stops there. All kinds start at
# N=100 so curves share the same x-axis start. The largest N in each list also
# serves as the ethalon for that (resolution, kind).
N_VALUES = {
    'real':   {256:  [100, 150, 200, 250, 300, 360],
               512:  [100, 150, 200, 250, 300, 360],
               1024: [100, 150, 200, 250, 300, 360]},
    'san':    {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
    'diffit': {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
}

# Diffit and san were trained on the same 3 classes but indexed them in
# different orders, so each generator needs its own class_map. Verified
# against 4_grid_plot.ipynb's GEN_NAME_FOR_PER_MODE.
CLASS_MAP_PER_KIND = {
    'san':    {'class_0': 'class_Ultra_Co25',
               'class_1': 'class_Ultra_Co11',
               'class_2': 'class_Ultra_Co6_2'},
    'diffit': {'class_0': 'class_Ultra_Co11',
               'class_1': 'class_Ultra_Co25',
               'class_2': 'class_Ultra_Co6_2'},
}

types_dict = {
    'Ultra_Co11':  'small grain',
    'Ultra_Co25':  'medium grain',
    'Ultra_Co8':   'medium-small grain',
    'Ultra_Co6_2': 'large grain',
    'Ultra_Co15':  'medium-small grain',
}

# Plot config — columns of the 3x3 grid (the 3 grain classes diffit was trained on).
CLASSES = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']
GRAIN_LABEL = {'class_Ultra_Co11':  'small grain',
               'class_Ultra_Co25':  'medium grain',
               'class_Ultra_Co6_2': 'large grain'}
LABELS = {'real': 'original', 'san': 'san', 'diffit': 'diffit '}

# Stats config (used in the convergence + gain-distribution sections).
METRICS = ['w_dist', 'mu1', 'mu2', 'sigma1', 'sigma2', 'amp1', 'amp2']
KINDS = ['san', 'diffit']
ENDPOINTS_BY_KIND = {'real': (100, 300), 'san': (360, 10000), 'diffit': (360, 10000)}
KIND_DISPLAY = {'real': 'Real', 'san': 'SAN', 'diffit': 'DiffiT'}

# Generation

For every (resolution, source) in `SOURCES`, extract angle distributions at every N in `N_VALUES[kind]`. Output: `./data/angles/<h5-stem>_msl5/angles_n{N}.parquet`.

The check is **step-aware**: a parquet is up-to-date only if it contains rows tagged with the current `STEP`. Changing `STEP` and re-running overwrites mismatched files (`generate_angles` does not append). The largest N per source doubles as the ethalon for the convergence plot. For san at 100k images this is the slowest step; originals reuse existing `prep_cache_*_n360.npy` caches so N=360 is fast.

In [ ]:
GEN_KW = dict(workers=20, angles_tol=3, min_segment_len=5.0,
              keep_contours=False, chunksize=64)

for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        data.generate_angles_sweep(
            h5_path, sweep_dir(h5_path),
            ns=N_VALUES[kind][resolution], step=STEP,
            types_dict=types_dict, tag=kind,
            run_meta={'family': kind, 'resolution': resolution, 'tags': [], 'notes': ''},
            **GEN_KW,
        )

# Convergence grid

Walk every (resolution, kind) sweep folder once, compare each sample-size parquet against its resolution's ethalon (largest-N real parquet), and collect:

- `records_by_panel[(resolution, kind)]` — feeds the 3×3 plot below (rows = resolution, cols = grain class).
- `df_metrics` — long-form table for the statistical tests in the next section.

The 1024 row currently has only the original baseline (no san/diffit h5s at 1024). Real self-vs-ethalon at N_max is exactly zero by construction; it's dropped so the log-y axis stays anchored.

In [ ]:
def collect_records():
    """Walk SOURCES once and return (records_by_panel, df_metrics)."""
    records_by_panel, rows = {}, []
    for resolution, group in SOURCES.items():
        ethalon = find_ethalon(sweep_dir(group['real']))
        if ethalon is None:
            print(f'[row {resolution}] no ethalon parquet — skip row')
            continue
        print(f'row {resolution}: ethalon = {ethalon}')
        for kind, h5 in group.items():
            d = sweep_dir(h5)
            if not d.exists():
                continue
            recs = metrics_vs_n(d, str(ethalon),
                                class_map=CLASS_MAP_PER_KIND.get(kind, {}),
                                step=STEP,
                                allowed_ns=set(N_VALUES[kind][resolution]))
            # Real self-vs-ethalon at N_max == 0 is structural — drop so log-y
            # plots stay sensible and stats don't see a trivial zero.
            if kind == 'real':
                recs = [r for r in recs if r['w_dist'] > 0]
            records_by_panel[(resolution, kind)] = recs
            rows.extend({'kind': kind, 'resolution': resolution, **r} for r in recs)
    return records_by_panel, pd.DataFrame.from_records(rows)

records_by_panel, df_metrics = collect_records()

SAVE_PATH = f'wdist_convergence_step{STEP}.png'
fig = plot_wdist_convergence_grid(
    records_by_panel, classes=CLASSES,
    kind_labels=LABELS, grain_labels=GRAIN_LABEL,
    row_keys=list(SOURCES), save_path=SAVE_PATH,
)
print(f'saved: {SAVE_PATH}')
fig.show()

# Statistical tests: do metrics improve with N?

Per (kind, resolution, class) curve, test whether the metric magnitude decreases with N using a one-sided Kendall τ on `(N, |metric|)` (`alternative='less'`). For `w_dist` (already non-negative) we use the raw value; for signed Gaussian-fit relatives (`mu1/mu2/sigma1/sigma2/amp1/amp2`) we use absolute value, since closer to 0 = closer to the ethalon.

**`rel_err_abs_%`** = `(|m|@N_lo − |m|@N_hi) / |m|@N_lo × 100` — positive = |m| shrank between endpoints. Endpoints are kind-specific:

- **real**: sweep is `[100, 150, 200, 250, 300, 360]`; N=360 self-vs-ethalon is zero and dropped → 5 points, **N_lo=100, N_hi=300**.
- **san / diffit**: 8 points, **N_lo=1000, N_hi=10000** (the regime where convergence flattens).

Aggregations combine one-sided p-values via Fisher's method: per (kind, resolution), per (kind, metric), per metric, per kind. Each row carries `mean_rel_err_abs_%` so significance and effect size sit side by side.

In [ ]:
result = convergence_stats(df_metrics, METRICS, ENDPOINTS_BY_KIND,
                           expected_points={'real': 5, 'san': 8, 'diffit': 8})

print_convergence_report(result, METRICS, KINDS, ENDPOINTS_BY_KIND,
                         step=STEP, kind_display=KIND_DISPLAY)

# Gain distribution

Per-kind distribution of `gain_pct` (sampling-only gain remaining from N_max to the plateau, in %), colored by resolution. Companion to the per-curve report above: shows the generator-wide spread of how much room is left for sampling alone to close the gap to the bias floor.

In [ ]:
STEP_PCT = 1.5  # bin width in percentage points
RES_STYLE = {256: {'color': 'rgba(31,119,180,0.55)', 'label': '256x256'},
             512: {'color': 'rgba(255,127,14,0.95)', 'label': '512x512'}}

summary = summarize_metric_distribution(result, 'gain_pct', KINDS, list(RES_STYLE))
for tag, line in summary.items():
    print(f'  {tag}:  {line}')

SAVE_PATH = f'gain_dist_step{STEP}_binstep{int(STEP_PCT)}.png'
fig = plot_metric_distribution(
    result, metric='gain_pct', kinds=KINDS, resolutions=list(RES_STYLE),
    kind_display=KIND_DISPLAY, res_style=RES_STYLE,
    bin_step=STEP_PCT, x_range=(-20, 60),
    save_path=SAVE_PATH,
    png_meta={
        'Title':       'gain_% distribution per generator and resolution',
        'angle_step':  f'{STEP}', 'bin_step_%': f'{STEP_PCT}',
        'kinds':       ','.join(KINDS),
        'resolutions': ','.join(str(r) for r in RES_STYLE),
        'endpoints':   ' | '.join(f'{k}:{ENDPOINTS_BY_KIND[k][0]}-{ENDPOINTS_BY_KIND[k][1]}'
                                  for k in KINDS),
        'fit_model':   '|m|(N) = a + b * N^(-1/2)',
        'stats':       ' | '.join(f'{k}: {v}' for k, v in summary.items()),
        'Software':    'plotly',
    },
)
print(f'saved: {SAVE_PATH}')
fig.show()